# VQA with CLIP and Attention Mechanism

## Description
This notebook implements a Visual Question Answering (VQA) model that leverages the power of CLIP for robust image and text representations, combined with an attention mechanism to focus on relevant image regions.

### Architecture:
1.  **Image Encoder**: The CLIP Vision Transformer (ViT) is used to extract patch-level features from the input image. These features serve as the input for the attention mechanism.
2.  **Text Encoder**: The CLIP Text Transformer encodes the input question into a fixed-size vector.
3.  **Attention**: An attention network takes the question vector and the image patch features to compute attention weights, producing an attention-weighted image vector that focuses on the most relevant parts of the image.
4.  **Classifier**: The attention-weighted image vector is combined with the question vector and fed into a classifier to predict the final answer.

## Workflow:
1.  **Setup**: Import libraries, load configurations, and set up the CLIP model.
2.  **Data Loading**: Create a custom PyTorch Dataset that processes raw images and questions using the CLIP processor.
3.  **Model Definition**: Define the new `CLIPAttentionVQAModel` PyTorch module.
4.  **Training**: Implement a training loop to train the model on the VQA dataset.
5.  **Evaluation**: Evaluate the model's performance on the validation set.
6.  **Visualization & Demo**: Plot training results and test the model with sample images and questions.

## 1. Import Libraries and Setup

In [ ]:
import os
import sys
import json
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import time
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# Install and import CLIP
try:
    import clip
    print("✅ CLIP is already installed.")
except ImportError:
    print("📦 Installing CLIP...")
    # !pip install ftfy regex tqdm
    # !pip install git+https://github.com/openai/CLIP.git
    import clip
    print("✅ CLIP installed successfully.")

# Add project path to resolve module imports
project_path = './VizWiz-VQA-PyTorch-master'
if project_path not in sys.path:
    sys.path.append(project_path)

try:
    import models
    import utils
    print("✅ Project modules (models, utils) imported successfully.")
except ImportError as e:
    print(f"❌ Error importing project modules: {e}")
    print("Please ensure the 'VizWiz-VQA-PyTorch-master' directory is in the correct location.")
    raise

print("\n🚀 VQA with CLIP and Attention - Setup Complete")
print("=" * 50)

## 2. Configuration and CLIP Model Loading

In [ ]:
# Load config
config_path = os.path.join(project_path, 'config/default.yaml')
with open(config_path, 'r') as handle:
    config = yaml.load(handle, Loader=yaml.FullLoader)

# --- Key Configuration Updates for this Model ---
# Use raw images, not pre-extracted features
config['images']['mode'] = 'raw' 
# Set attention glimpses
config['model']['attention']['glimpses'] = 2
# Update paths
config['annotations']['dir'] = './data/Annotations'
config['images']['dir'] = './data'
config['annotations']['path_vocabs'] = './prepro_data/vocabs.json'
# Add missing training configuration
if 'training' not in config:
    config['training'] = {}
config['training']['filter_unanswerable'] = False  # Don't filter unanswerable questions

print("✅ Configuration loaded and updated for the CLIP-Attention model.")

# Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
cudnn.benchmark = True
print(f"🔧 Using device: {device}")

# Load CLIP model and preprocessor
print("\n📥 Loading CLIP model (ViT-B/32)...")
# For this architecture, we need the model to output the sequence of patch embeddings, not just the pooled output.
# We will load the standard model and then modify its forward pass or hook into it.
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()  # Set to evaluation mode

# Get model dimensions
clip_visual_width = clip_model.visual.class_embedding.shape[0]
clip_output_dim = clip_model.visual.output_dim
print(f"📐 CLIP Visual Patch Dimension: {clip_visual_width}")
print(f"📐 CLIP Final Output Dimension: {clip_output_dim}")
print("✅ CLIP model loaded successfully.")

## 3. Custom Dataset for Raw Images
The standard dataset loader uses pre-extracted HDF5 features. We need a new one that loads raw images from disk and processes them with CLIP's preprocessor on the fly.

In [ ]:
class VQADatasetWithCLIP(Dataset):
    def __init__(self, config, split, clip_processor, vocabs):
        self.config = config
        self.split = split
        self.clip_processor = clip_processor
        self.vocabs = vocabs
        self.answer_vocab = vocabs['answer']
        self.question_vocab = vocabs['question']
        self.num_tokens = len(self.question_vocab)
        
        # Load annotations with UTF-8 encoding
        ann_path = os.path.join(config['annotations']['dir'], f"{split}.json")
        with open(ann_path, 'r', encoding='utf-8') as f:
            self.annotations = json.load(f)
            
        # Filter out unanswerable questions if needed (with safe check)
        filter_unanswerable = config.get('training', {}).get('filter_unanswerable', False)
        if filter_unanswerable:
            self.annotations = [ann for ann in self.annotations if ann.get('answerable', True)]

        self.img_dir = os.path.join(config['images']['dir'], split)
        print(f"✅ VQADatasetWithCLIP for '{split}' split initialized with {len(self.annotations)} samples.")

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        
        # --- Image Processing ---
        img_path = os.path.join(self.img_dir, ann['image'])
        try:
            image = Image.open(img_path).convert('RGB')
            processed_image = self.clip_processor(image)
        except FileNotFoundError:
            print(f"⚠️ Image not found: {img_path}. Returning a dummy tensor.")
            processed_image = torch.zeros((3, 224, 224))

        # --- Question Processing ---
        question_tokens = [self.question_vocab.get(word, self.question_vocab['<unk>']) for word in ann['question'].split()]
        q_len = len(question_tokens)
        # Pad question - get max length with safe fallback
        max_q_len = self.config.get('model', {}).get('seq2vec', {}).get('max_length', 20)
        if q_len > max_q_len:
            question_tokens = question_tokens[:max_q_len]
            q_len = max_q_len
        else:
            question_tokens.extend([self.question_vocab['<pad>']] * (max_q_len - q_len))
        
        question = torch.tensor(question_tokens, dtype=torch.long)

        # --- Answer Processing ---
        # Create a multi-label answer tensor
        answer = torch.zeros(len(self.answer_vocab))
        if 'answers' in ann and ann['answers']:
            # Use the first answer for simplicity in this setup, or implement full VQA accuracy later
            ans = ann['answers'][0]['answer']
            if ans in self.answer_vocab:
                answer[self.answer_vocab[ans]] = 1
        
        return {
            'image': processed_image,
            'question': question,
            'q_length': torch.tensor(q_len, dtype=torch.long),
            'answer': answer
        }

def collate_fn_clip(batch):
    # Custom collate function to handle the dictionary output of our new dataset
    images = torch.stack([item['image'] for item in batch])
    questions = torch.stack([item['question'] for item in batch])
    q_lengths = torch.stack([item['q_length'] for item in batch])
    answers = torch.stack([item['answer'] for item in batch])
    
    return {
        'image': images,
        'question': questions,
        'q_length': q_lengths,
        'answer': answers
    }

# Load vocabularies with UTF-8 encoding
with open(config['annotations']['path_vocabs'], 'r', encoding='utf-8') as f:
    vocabs = json.load(f)

print(f"📖 Vocabularies loaded. Question vocab size: {len(vocabs['question'])}, Answer vocab size: {len(vocabs['answer'])}")

## 4. Define the CLIP-Attention VQA Model
This is the core of our new architecture.

In [ ]:
class CLIPAttentionVQAModel(nn.Module):
    def __init__(self, config, clip_model, num_tokens, num_answers):
        super().__init__()
        self.config = config
        
        # --- CLIP Encoders ---
        self.clip_visual_encoder = clip_model.visual
        self.clip_text_encoder = clip_model.transformer # Using the text transformer directly
        
        # Freeze CLIP parameters
        for param in self.clip_visual_encoder.parameters():
            param.requires_grad = False
        for param in self.clip_text_encoder.parameters():
            param.requires_grad = False
            
        # --- VQA-specific components ---
        dim_q = config['model']['pooling']['dim_q']
        dim_v_att = self.clip_visual_encoder.output_dim # Dimension of each patch embedding
        
        # Question encoder (can be a simple embedding layer or a more complex one)
        self.text_encoder = models.TextEncoder(
            num_tokens=num_tokens,
            emb_size=config['model']['seq2vec']['emb_size'],
            dim_q=dim_q,
            drop=config['model']['seq2vec']['dropout'],
        )
        
        # Attention mechanism
        self.attention = models.Attention(
            dim_v=dim_v_att,
            dim_q=dim_q,
            dim_h=config['model']['attention']['mid_features'],
            n_glimpses=config['model']['attention']['glimpses'],
            drop=config['model']['attention']['dropout'],
        )
        
        # Classifier
        self.classifier = models.Classifier(
            dim_input=config['model']['attention']['glimpses'] * dim_v_att + dim_q,
            dim_h=config['model']['pooling']['dim_h'],
            top_ans=num_answers,
            drop=config['model']['classifier']['dropout'],
        )
        print("✅ CLIPAttentionVQAModel initialized.")

    def forward(self, image, question, q_length):
        # --- Get Image Features from CLIP ---
        # To get patch features, we need to bypass the final projection and pooling of the ViT
        # A common way is to hook into the last transformer block
        x = self.clip_visual_encoder.conv1(image.type(self.clip_visual_encoder.conv1.weight.dtype))
        x = x.reshape(x.shape[0], x.shape[1], -1)  # shape = [*, width, grid ** 2]
        x = x.permute(0, 2, 1)  # shape = [*, grid ** 2, width]
        x = torch.cat([self.clip_visual_encoder.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device), x], dim=1)  # shape = [*, grid ** 2 + 1, width]
        x = x + self.clip_visual_encoder.positional_embedding.to(x.dtype)
        x = self.clip_visual_encoder.ln_pre(x)

        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.clip_visual_encoder.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD
        
        # x now contains the sequence of patch embeddings + class embedding
        # We can use all of them for attention
        v = self.clip_visual_encoder.ln_post(x[:, 1:, :]) # Exclude the [CLS] token
        v = v.permute(0, 2, 1) # N, L, D -> N, D, L
        # Reshape to be compatible with attention mechanism (N, D, H, W)
        # The number of patches is L, let's say L = H*W
        L = v.shape[2]
        H = W = int(np.sqrt(L))
        v = v.reshape(v.shape[0], v.shape[1], H, W)

        # --- Get Question Features ---
        q = self.text_encoder(question, list(q_length.data))
        
        # --- Attention ---
        v = F.normalize(v, p=2, dim=1) # Normalize visual features
        att_maps = self.attention(v, q)
        v_att = models.apply_attention(v, att_maps)
        
        # --- Combine and Classify ---
        combined = torch.cat([v_att, q], dim=1)
        answer = self.classifier(combined)
        
        return answer


## 5. Initialize Datasets, Dataloaders, and Model

In [ ]:
# Optimize configuration for faster training
print("⚡ Optimizing configuration for speed...")

# Increase batch size if GPU memory allows
original_batch_size = config['training']['batch_size']
if torch.cuda.is_available():
    # Try to use larger batch size for better GPU utilization
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3  # GB
    if gpu_memory > 8:  # If GPU has more than 8GB
        config['training']['batch_size'] = min(32, original_batch_size * 2)
    print(f"🔧 GPU Memory: {gpu_memory:.1f}GB, Batch size: {original_batch_size} → {config['training']['batch_size']}")

# Reduce data workers if on Windows to avoid multiprocessing issues
if os.name == 'nt':  # Windows
    config['training']['data_workers'] = min(2, config['training']['data_workers'])
    print(f"🪟 Windows detected, reducing data workers to {config['training']['data_workers']}")

# Create datasets
print("📚 Creating datasets...")
train_dataset = VQADatasetWithCLIP(config, 'train', clip_preprocess, vocabs)
val_dataset = VQADatasetWithCLIP(config, 'val', clip_preprocess, vocabs)

print(f"📊 Dataset sizes:")
print(f"  - Train: {len(train_dataset):,} samples")
print(f"  - Val: {len(val_dataset):,} samples")

# Create dataloaders
print("📦 Creating dataloaders...")
train_loader = DataLoader(
    train_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=config['training']['data_workers'],
    pin_memory=True,
    collate_fn=collate_fn_clip,
    persistent_workers=True if config['training']['data_workers'] > 0 else False
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=config['training']['data_workers'],
    pin_memory=True,
    collate_fn=collate_fn_clip,
    persistent_workers=True if config['training']['data_workers'] > 0 else False
)
print(f"  - Train loader: {len(train_loader)} batches ({len(train_loader) * config['training']['batch_size']} samples)")
print(f"  - Val loader: {len(val_loader)} batches ({len(val_loader) * config['training']['batch_size']} samples)")

# Calculate estimated time
samples_per_second_estimate = 10  # Conservative estimate for CLIP processing
estimated_time_per_epoch = len(train_dataset) / samples_per_second_estimate / 60  # minutes
print(f"⏱️ Estimated time per epoch: ~{estimated_time_per_epoch:.1f} minutes")

# Initialize model
print("\n🤖 Initializing CLIP-Attention VQA Model...")
model = CLIPAttentionVQAModel(
    config,
    clip_model,
    num_tokens=train_dataset.num_tokens,
    num_answers=len(vocabs['answer'])
).to(device)

# Use DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"🔥 Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

# Print model statistics
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Model Statistics:")
print(f"  - Total parameters: {total_params:,}")
print(f"  - Trainable parameters: {trainable_params:,}")
print(f"  - Frozen parameters: {total_params - trainable_params:,}")

# Test a small batch to estimate actual speed
print("\n🧪 Testing batch processing speed...")
test_batch = next(iter(train_loader))
start_time = time.time()
with torch.no_grad():
    test_images = test_batch['image'][:4].to(device)  # Test with 4 samples
    test_questions = test_batch['question'][:4].to(device)
    test_q_lengths = test_batch['q_length'][:4].to(device)
    _ = model(test_images, test_questions, test_q_lengths)
test_time = time.time() - start_time
samples_per_second = 4 / test_time
print(f"  - Processing speed: ~{samples_per_second:.1f} samples/second")
print(f"  - Revised time estimate per epoch: ~{len(train_dataset) / samples_per_second / 60:.1f} minutes")

## 6. Training and Evaluation Loop

In [ ]:
def run_epoch(model, dataloader, is_train, optimizer, device, epoch_num):
    """A generic function to run one epoch of training or evaluation."""
    if is_train:
        model.train()
        prefix = "Train"
    else:
        model.eval()
        prefix = "Val"

    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    # Add timing information
    batch_times = []
    data_times = []
    
    pbar = tqdm(dataloader, desc=f'Epoch {epoch_num+1} - {prefix}')
    
    with torch.set_grad_enabled(is_train):
        data_start = time.time()
        
        for batch_idx, batch in enumerate(pbar):
            data_time = time.time() - data_start
            data_times.append(data_time)
            
            batch_start = time.time()
            
            images = batch['image'].to(device, non_blocking=True)
            questions = batch['question'].to(device, non_blocking=True)
            q_lengths = batch['q_length'].to(device, non_blocking=True)
            answers = batch['answer'].to(device, non_blocking=True)

            outputs = model(images, questions, q_lengths)
            
            # Using BCEWithLogitsLoss for multi-label classification
            loss = F.binary_cross_entropy_with_logits(outputs, answers)
            
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                # Add gradient clipping to stabilize training
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            # Calculate accuracy (top-1 for simplicity)
            preds = torch.sigmoid(outputs)
            _, predicted_idx = preds.max(1)
            _, answer_idx = answers.max(1)
            correct = (predicted_idx == answer_idx).sum().item()

            total_loss += loss.item() * images.size(0)
            total_correct += correct
            total_samples += images.size(0)
            
            batch_time = time.time() - batch_start
            batch_times.append(batch_time)
            
            # Update progress bar with more detailed info
            avg_batch_time = np.mean(batch_times[-10:])  # Average of last 10 batches
            avg_data_time = np.mean(data_times[-10:])
            samples_per_sec = images.size(0) / avg_batch_time
            
            pbar.set_postfix({
                'Loss': f'{total_loss/total_samples:.4f}',
                'Acc': f'{total_correct/total_samples:.4f}',
                'Speed': f'{samples_per_sec:.1f}s/s',
                'Data': f'{avg_data_time:.2f}s',
                'Batch': f'{avg_batch_time:.2f}s'
            })
            
            # Early stopping for testing (remove this for full training)
            # if batch_idx >= 5:  # Only run 5 batches for testing
            #     print(f"\n⚠️ Early stopping after {batch_idx+1} batches for testing")
            #     break
                
            data_start = time.time()
            
    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    avg_batch_time = np.mean(batch_times)
    avg_data_time = np.mean(data_times)
    
    print(f"\n{prefix} Stats:")
    print(f"  - Avg batch time: {avg_batch_time:.2f}s")
    print(f"  - Avg data loading time: {avg_data_time:.2f}s")
    print(f"  - Samples per second: {total_samples / sum(batch_times):.1f}")
    
    return avg_loss, avg_acc

# --- Training Setup ---
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=config['training']['lr'],
    weight_decay=1e-4  # Add weight decay for regularization
)

# Add learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2, verbose=True
)

# Create log directory
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_dir = f'./logs/clip_attention/clip_attention_{timestamp}'
os.makedirs(log_dir, exist_ok=True)
best_model_path = os.path.join(log_dir, 'best_model.pth')
print(f"📁 Logs and models will be saved to: {log_dir}")

# --- Main Loop ---
best_val_acc = 0
training_log = []

print("\n🚀 Starting Training...")
print(f"📋 Training Configuration:")
print(f"  - Epochs: {config['training']['epochs']}")
print(f"  - Batch Size: {config['training']['batch_size']}")
print(f"  - Learning Rate: {config['training']['lr']}")
print(f"  - Device: {device}")
print("=" * 50)

total_training_start = time.time()

for epoch in range(config['training']['epochs']):
    epoch_start_time = time.time()
    
    print(f"\n🔄 Epoch {epoch+1}/{config['training']['epochs']}")
    
    # Train
    train_loss, train_acc = run_epoch(model, train_loader, True, optimizer, device, epoch)
    
    # Validate
    val_loss, val_acc = run_epoch(model, val_loader, False, None, device, epoch)
    
    # Update learning rate
    scheduler.step(val_acc)
    
    epoch_time = time.time() - epoch_start_time
    
    print(f"\n📊 Epoch {epoch+1} Summary:")
    print(f"  - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  - Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print(f"  - Time: {epoch_time:.1f}s ({epoch_time/60:.1f}min)")
    print(f"  - LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    log_entry = {
        'epoch': epoch + 1,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss, 'val_acc': val_acc,
        'time': epoch_time,
        'lr': optimizer.param_groups[0]['lr']
    }
    training_log.append(log_entry)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"  ⭐ New best validation accuracy! Saving model to {best_model_path}")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_acc': val_acc,
            'config': config
        }, best_model_path)
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        checkpoint_path = os.path.join(log_dir, f'checkpoint_epoch_{epoch+1}.pth')
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_acc': val_acc,
            'config': config
        }, checkpoint_path)
        print(f"  💾 Checkpoint saved: {checkpoint_path}")

total_training_time = time.time() - total_training_start

print(f"\n✅ Training complete!")
print(f"🏆 Best Validation Accuracy: {best_val_acc:.4f}")
print(f"⏱️ Total Training Time: {total_training_time/3600:.2f} hours")

# Save training log
with open(os.path.join(log_dir, 'training_log.json'), 'w') as f:
    json.dump(training_log, f, indent=2)
print(f"📝 Training log saved.")

## 7. Visualize Training Results

In [ ]:
# Plot training curves
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

epochs = [log['epoch'] for log in training_log]
train_losses = [log['train_loss'] for log in training_log]
val_losses = [log['val_loss'] for log in training_log]
train_accs = [log['train_acc'] for log in training_log]
val_accs = [log['val_acc'] for log in training_log]

# Loss curves
ax1.plot(epochs, train_losses, 'o-', label='Training Loss')
ax1.plot(epochs, val_losses, 'o-', label='Validation Loss')
ax1.set_title('Training and Validation Loss', fontsize=16)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

# Accuracy curves
ax2.plot(epochs, train_accs, 'o-', label='Training Accuracy')
ax2.plot(epochs, val_accs, 'o-', label='Validation Accuracy')
ax2.set_title('Training and Validation Accuracy', fontsize=16)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(log_dir, 'training_curves.png'))
plt.show()

## 8. Demo with a Real Image

In [ ]:
def demo(model, image_url, question, vocabs, clip_processor, device):
    """Demo function to test the model with a live image URL."""
    try:
        # Load and process image
        response = requests.get(image_url)
        image = Image.open(BytesIO(response.content)).convert("RGB")
        processed_image = clip_processor(image).unsqueeze(0).to(device)

        # Process question
        q_vocab = vocabs['question']
        q_tokens = [q_vocab.get(w, q_vocab['<unk>']) for w in question.lower().split()]
        q_len = len(q_tokens)
        max_len = config['model']['seq2vec']['max_length']
        if q_len > max_len:
            q_tokens = q_tokens[:max_len]
            q_len = max_len
        else:
            q_tokens.extend([q_vocab['<pad>']] * (max_len - q_len))
        
        processed_question = torch.tensor(q_tokens, dtype=torch.long).unsqueeze(0).to(device)
        q_length = torch.tensor([q_len], dtype=torch.long).to(device)

        # --- Model Prediction ---
        model.eval()
        with torch.no_grad():
            output = model(processed_image, processed_question, q_length)
            probs = torch.sigmoid(output).squeeze()
        
        # Get top 5 answers
        top_probs, top_indices = torch.topk(probs, 5)
        idx_to_ans = {v: k for k, v in vocabs['answer'].items()}

        # --- Display Results ---
        plt.imshow(image)
        plt.axis('off')
        plt.title(f"Q: {question}")
        plt.show()

        print("🤖 Top 5 Predicted Answers:")
        for i in range(5):
            prob = top_probs[i].item()
            ans = idx_to_ans.get(top_indices[i].item(), "<unknown>")
            print(f"  {i+1}. {ans} (Confidence: {prob:.4f})")

    except Exception as e:
        print(f"❌ Demo failed: {e}")

# --- Load the best model for the demo ---
print("\n📥 Loading best trained model for demo...")
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Best model from epoch {checkpoint['epoch']} with accuracy {checkpoint['val_acc']:.4f} loaded.")

# --- Run Demo ---
demo_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
demo_question = "what is on the table?"
demo(model, demo_url, demo_question, vocabs, clip_preprocess, device)

demo_url_2 = "https://images.unsplash.com/photo-1543466835-00a7907e9de1?w=500"
demo_question_2 = "what kind of animal is this?"
demo(model, demo_url_2, demo_question_2, vocabs, clip_preprocess, device)